# Tokenisateurs comparés — WordPiece et Unigram, les deux autres familles de sous-mots

**Navigation** : [Version Python](04b-Tokenisateurs-Compares.ipynb) · [04 Tokenisation](04-Tokenisation-From-Scratch.ipynb) · [RAG et mémoire sémantique](README.md)

**Objectifs** : construire BPE, WordPiece et Unigram, comparer leurs coûts, les confronter aux entraîneurs réels puis mesurer trois traductions. **Prérequis** : .NET Interactive, Python partagé et environnement isolé avec `tokenizers` et `huggingface-hub` ; accès aux petits fichiers publics de tokenisation lors du premier lancement. Aucun poids de modèle, GPU ou clé. **Durée indicative** : 10 minutes. La recette portable de préparation est dans le [README](README.md).

La BCL réalise les algorithmes pédagogiques ; Python.NET appelle les entraîneurs Rust `tokenizers` et le pipeline DistilBERT. Qwen est calculé par Microsoft.ML.Tokenizers en .NET. Les budgets de vocabulaire sont **voisins** : 94 symboles BPE/WP et 93 pièces Unigram.

Les avertissements CS1701 éventuels signalent les versions de références entre .NET Interactive, les bibliothèques et le runtime ; ils restent visibles dans les sorties.


In [1]:
#r "nuget: Microsoft.ML.Tokenizers, 3.0.0"
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Text.Json;
using System.Text.RegularExpressions;
using System.Globalization;
using System.Security.Cryptography;
using System.IO;
using Microsoft.ML.Tokenizers;
using Microsoft.DotNet.Interactive.Formatting;
using Python.Runtime;
Console.WriteLine($".NET {Environment.Version} ; Microsoft.ML.Tokenizers 3.0.0 ; pythonnet 3.0.5");

Installed Packages Microsoft.ML.Tokenizers, 3.0.0 pythonnet, 3.0.5

.NET 10.0.14 ; Microsoft.ML.Tokenizers 3.0.0 ; pythonnet 3.0.5


## 1. Objectif — le critère de sélection fait la famille

BPE sélectionne la paire la plus fréquente ; WordPiece pédagogique privilégie la cohésion `freq(ab)/(freq(a)×freq(b))` sous un plancher de fréquence ; Unigram part de candidats et élague après segmentation probabiliste. Le corpus et les budgets permettent d'isoler les effets de ces choix.

## 2. Le même corpus, le même BPE de référence

Les fables de La Fontaine sont découpées exactement comme en Python. Le marqueur de fin de mot `</w>` est retiré après encodage ; un marqueur resté seul ne compte pas comme token.

In [2]:
string fables = @"Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.
Maître Renard, par l'odeur alléché, lui tint à peu près ce langage.
Hé bonjour, Monsieur du Corbeau. Que vous êtes joli, que vous me semblez beau.
Sans mentir, si votre ramage se rapporte à votre plumage, vous êtes le Phénix des hôtes de ces bois.
Le corbeau, honteux et confus, jura, mais un peu tard, qu'on ne l'y prendrait plus.
La cigale ayant chanté tout l'été, se trouva fort dépourvue quand la bise fut venue.
Pas un seul petit morceau de mouche ou de vermisseau, elle alla crier famine chez la fourmi sa voisine.
Je vous paierai, lui dit-elle, avant l'août, foi d'animal, intérêt et principal.
La fourmi n'est pas prêteuse, c'est là son moindre défaut.
Que faisiez-vous au temps chaud, dit-elle à cette emprunteuse.
Nuit et jour à tout venant je chantais, ne vous déplaise.
Vous chantiez, j'en suis fort aise, eh bien dansez maintenant.
Le lièvre et la tortue firent une course, et la tortue, lente mais obstinée,
gagna contre le lièvre trop confiant.
Rien ne sert de courir, il faut partir à point, la patience vaut mieux que la force.
Le lion, roi des animaux, apprit un jour que les bêtes de son royaume le craignaient.
Le rat des villes et le rat des champs se rencontrèrent et partagèrent leur repas.
Le loup et l'agneau se désaltéraient au même ruisseau, le loup chercha une querelle à l'agneau.
Le corbeau et le renard se parlèrent longtemps, le renard eut le fromage.
Le chêne et le roseau se disputaient la force, le vent les départagea.
Perrette et le pot au lait, la poule aux œufs d'or, le coq et le renard.
Le savetier et le financier, le sage et le fou, le laboureur et ses enfants.";
string[] Words(string s)=>Regex.Matches(s.ToLowerInvariant(),"[a-zà-ÿ'éèêëïôöûüç0-9-]+").Select(m=>m.Value).ToArray();
var words=Words(fables);var chars=words.SelectMany(w=>w.Select(c=>c.ToString())).Append("</w>").ToHashSet();
List<(string a,string b)> Train(string[] input,int budget,bool cohesion){
 var data=input.Select(w=>w.Select(c=>c.ToString()).Append("</w>").ToList()).ToList();var merges=new List<(string,string)>();
 for(int step=0;step<budget;step++){var pairs=new Dictionary<(string,string),int>();var singles=new Dictionary<string,int>();foreach(var w in data){foreach(var t in w)singles[t]=singles.GetValueOrDefault(t)+1;for(int j=0;j+1<w.Count;j++){var pair=(w[j],w[j+1]);pairs[pair]=pairs.GetValueOrDefault(pair)+1;}}
 var eligible=pairs.Where(kv=>kv.Value>=2).ToList();if(eligible.Count==0)break;
 var best=eligible.OrderByDescending(kv=>cohesion?(double)kv.Value/(singles[kv.Key.Item1]*singles[kv.Key.Item2]):kv.Value).ThenByDescending(kv=>kv.Key.Item1,StringComparer.Ordinal).ThenByDescending(kv=>kv.Key.Item2,StringComparer.Ordinal).First().Key;merges.Add(best);
 for(int j=0;j<data.Count;j++){var old=data[j];var next=new List<string>();for(int z=0;z<old.Count;z++){if(z+1<old.Count && (old[z],old[z+1])==best){next.Add(old[z]+old[z+1]);z++;}else next.Add(old[z]);}data[j]=next;}}
 return merges;
}
string[] Encode(string word,List<(string a,string b)> merges){var syms=word.Select(c=>c.ToString()).Append("</w>").ToList();var rank=merges.Select((p,i)=>(p,i)).ToDictionary(x=>x.p,x=>x.i);while(true){int best=int.MaxValue;for(int i=0;i+1<syms.Count;i++)if(rank.TryGetValue((syms[i],syms[i+1]),out int r)&&r<best)best=r;if(best==int.MaxValue)break;var pair=merges[best];var next=new List<string>();for(int i=0;i<syms.Count;i++){if(i+1<syms.Count&&(syms[i],syms[i+1])==pair){next.Add(syms[i]+syms[i+1]);i++;}else next.Add(syms[i]);}syms=next;}return syms.Select(s=>s.EndsWith("</w>")?s[..^4]:s).Where(s=>s.Length>0).ToArray();}
var bpe=Train(words,60,false);int bpeCount=words.Sum(w=>Encode(w,bpe).Length);Console.WriteLine($"Corpus : {words.Length} mots, {words.Distinct().Count()} distincts ; BPE {bpe.Count} fusions, {bpeCount} tokens, {bpeCount/(double)words.Length:F3} tokens/mot, vocabulaire {bpe.Count+chars.Count}.");


Corpus : 299 mots, 188 distincts ; BPE 60 fusions, 871 tokens, 2.913 tokens/mot, vocabulaire 94.



warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotNet.Interactive.Formatting' matches identity 'Microsoft.AspNetCore.Html.Abstractions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' of 'Microsoft.AspNetCore.Html.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.AspNetCore.Html.Abstractions, Version=2.3.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' used by 'Microsoft.DotN

### Lecture du résultat

Le corpus contient 299 mots et 188 types. Avec 60 fusions et 94 symboles, le BPE produit 871 tokens, soit 2,913 tokens/mot. Le marqueur pur est exclu du décompte, contrairement au carnet 04.

## 3. WordPiece — la fusion au score de cohésion

Le score privilégie les paires fréquentes relativement aux fréquences de leurs membres. Le plancher est de deux occurrences, et les ex æquo sont départagés lexicalement.

In [3]:
var wp=Train(words,60,true);int wpCount=words.Sum(w=>Encode(w,wp).Length);Console.WriteLine($"WordPiece : {wp.Count} fusions, {wpCount} tokens, {wpCount/(double)words.Length:F3} tokens/mot, vocabulaire {wp.Count+chars.Count} ; {bpe.Intersect(wp).Count()} fusions communes.");for(int i=0;i<Math.Min(12,Math.Min(bpe.Count,wp.Count));i++)Console.WriteLine($"{i+1,2} : BPE {bpe[i].a}+{bpe[i].b} | WP {wp[i].a}+{wp[i].b}");

WordPiece : 60 fusions, 1099 tokens, 3.676 tokens/mot, vocabulaire 94 ; 5 fusions communes.


 1 : BPE e+</w> | WP c+h


 2 : BPE t+</w> | WP h+é


 3 : BPE s+</w> | WP è+v


 4 : BPE l+e</w> | WP i+èv


 5 : BPE e+n | WP l+ièv


... (3 outputs left out)

 9 : BPE r+</w> | WP lièv+r


10 : BPE a+n | WP q+u


11 : BPE a+i | WP î+t


12 : BPE e+t</w> | WP ît+r


### Lecture du résultat

WordPiece produit 1099 tokens, soit 3,676 par mot, et partage 5 fusions avec BPE sur 60. Les trois premières paires WordPiece sont `c+h`, `h+é`, `è+v` ; BPE commence par des fins de mot.

In [4]:
string[] testWords={"corbeau","vous","chanté","prêteuse","renard","fromage","dépourvue","patience"};foreach(var w in testWords)Console.WriteLine($"{w,-12} BPE [{string.Join(" + ",Encode(w,bpe))}] | WP [{string.Join(" + ",Encode(w,wp))}]");

corbeau      BPE [c + orbeau] | WP [corb + e + a + u]


vous         BPE [vous] | WP [vous]


chanté       BPE [ch + an + t + é] | WP [chan + té]


prêteuse     BPE [pr + êt + eu + se] | WP [p + r + êt + e + u + s + e]


renard       BPE [renard] | WP [r + e + n + a + r + d]


fromage      BPE [f + ro + m + age] | WP [fromag + e]


dépourvue    BPE [dé + p + our + v + ue] | WP [dép + o + u + r + v + u + e]


patience     BPE [pa + ti + en + c + e] | WP [p + a + t + i + e + n + c + e]


### Lecture du résultat

BPE conserve `renard` entier, tandis que le WordPiece jouet le découpe en six caractères ; `chanté` est à l’inverse découpé en quatre pièces BPE et deux pièces WordPiece.

### Exercice 1 — WordPiece sur un autre corpus

Entraîner les deux méthodes sur un corpus choisi et comparer les douze premières fusions ; expliquer une fusion que la cohésion choisit plus tôt. Les deux fixtures vérifient le résultat sur des corpus distincts.

In [5]:
(List<(string a,string b)> bpe,List<(string a,string b)> wp)? ComparerCorpus(string[] corpus){ // TODO : entraîner les deux familles sur 18 fusions
 return null;
}
var fixture=ComparerCorpus(new[]{"abab","abab","caca","caca","caca"});if(fixture==null)Console.WriteLine("Exercice 1 à compléter.");else {bool Check(string[] x,(List<(string a,string b)> bpe,List<(string a,string b)> wp)? y)=>y!=null && y.Value.bpe!=null && y.Value.wp!=null && y.Value.bpe.SequenceEqual(Train(x,18,false)) && y.Value.wp.SequenceEqual(Train(x,18,true));string[] a={"abab","abab","caca","caca","caca"},b={"banane","banane","cabane","cabane","cabane"};Console.WriteLine(Check(a,fixture)&&Check(b,ComparerCorpus(b))?"Fixtures correctes : expliquez à présent une différence sur votre corpus.":"Fixtures incorrectes.");}

Exercice 1 à compléter.


## 4. Unigram — partir de trop grand, élaguer par usage

On compte tous les sous-mots de longueur au plus 6, pondérés par fréquence, puis on alterne segmentation Viterbi et réestimation dure des usages. Trois passes précèdent l'élagage à 93, puis deux passes suivent. Les caractères restent garantis. Cet élagage par usage simplifie l'entraînement Unigram réel.

In [6]:
var wordFreq=words.GroupBy(x=>x).ToDictionary(g=>g.Key,g=>g.Count());var candidates=new Dictionary<string,int>();foreach(var kv in wordFreq)for(int len=1;len<=6;len++)for(int i=0;i+len<=kv.Key.Length;i++){string piece=kv.Key.Substring(i,len);candidates[piece]=candidates.GetValueOrDefault(piece)+kv.Value;}
string[] Viterbi(string word,Dictionary<string,double> logp){int n=word.Length;double[] score=Enumerable.Repeat(double.NegativeInfinity,n+1).ToArray();int[] prev=new int[n+1];score[0]=0;for(int i=1;i<=n;i++)for(int j=Math.Max(0,i-6);j<i;j++){string part=word.Substring(j,i-j);if(logp.TryGetValue(part,out double v)&&score[j]+v>score[i]){score[i]=score[j]+v;prev[i]=j;}}if(double.IsNegativeInfinity(score[n]))return word.Select(c=>c.ToString()).ToArray();var result=new List<string>();for(int i=n;i>0;){int j=prev[i];result.Add(word.Substring(j,i-j));i=j;}result.Reverse();return result.ToArray();}
Dictionary<string,int> Refit(Dictionary<string,int> initial,int iterations){var counts=new Dictionary<string,int>(initial);for(int t=0;t<iterations;t++){double total=counts.Values.Sum();var logp=counts.ToDictionary(x=>x.Key,x=>Math.Log(x.Value/total));var usage=new Dictionary<string,int>();foreach(var wf in wordFreq)foreach(var piece in Viterbi(wf.Key,logp))usage[piece]=usage.GetValueOrDefault(piece)+wf.Value;foreach(char c in wordFreq.Keys.SelectMany(x=>x).Distinct())if(!usage.ContainsKey(c.ToString()))usage[c.ToString()]=1;counts=usage;}return counts;}
Dictionary<string,int> Prune(Dictionary<string,int> counts,int k){var kept=wordFreq.Keys.SelectMany(x=>x).Select(c=>c.ToString()).ToHashSet();foreach(var kv in counts.OrderByDescending(x=>x.Value).ThenBy(x=>x.Key,StringComparer.Ordinal)){if(kept.Count>=k)break;kept.Add(kv.Key);}return kept.ToDictionary(x=>x,x=>counts[x]);}
var fitted=Refit(candidates,3);var finalCounts=Refit(Prune(fitted,93),2);var ugLog=finalCounts.ToDictionary(kv=>kv.Key,kv=>Math.Log(kv.Value/(double)finalCounts.Values.Sum()));int ugCount=words.Sum(w=>Viterbi(w,ugLog).Length);Console.WriteLine($"Candidats : {candidates.Count} ; Unigram {finalCounts.Count} pièces, {ugCount} tokens, {ugCount/(double)words.Length:F3} tokens/mot.");foreach(var kv in finalCounts.OrderByDescending(x=>x.Value).ThenBy(x=>x.Key,StringComparer.Ordinal).Take(15))Console.WriteLine($"{kv.Key} : {kv.Value}");

Candidats : 1564 ; Unigram 93 pièces, 850 tokens, 2.843 tokens/mot.


r : 62


n : 52


e : 51


i : 44


a : 43


... (6 outputs left out)

c : 25


m : 22


et : 19


l : 19


### Lecture du résultat

Les 1564 candidates sont réduites à 93 pièces. Unigram produit 850 tokens, soit 2,843 par mot ; `r` est utilisé 62 fois et `le` 34 fois. Les caractères sont garantis même après élagage.

In [7]:
foreach(var w in testWords)Console.WriteLine($"{w,-12} BPE [{string.Join("+",Encode(w,bpe))}] | WP [{string.Join("+",Encode(w,wp))}] | UG [{string.Join("+",Viterbi(w,ugLog))}]");

corbeau      BPE [c+orbeau] | WP [corb+e+a+u] | UG [corbea+u]


vous         BPE [vous] | WP [vous] | UG [vous]


chanté       BPE [ch+an+t+é] | WP [chan+té] | UG [c+h+a+n+t+é]


prêteuse     BPE [pr+êt+eu+se] | WP [p+r+êt+e+u+s+e] | UG [p+r+ê+te+u+se]


renard       BPE [renard] | WP [r+e+n+a+r+d] | UG [renard]


fromage      BPE [f+ro+m+age] | WP [fromag+e] | UG [fromag+e]


dépourvue    BPE [dé+p+our+v+ue] | WP [dép+o+u+r+v+u+e] | UG [d+é+p+o+u+r+v+u+e]


patience     BPE [pa+ti+en+c+e] | WP [p+a+t+i+e+n+c+e] | UG [p+a+t+i+e+n+ce]


### Lecture du résultat

`corbeau` devient `c+orbeau` en BPE, `corb+e+a+u` en WordPiece et `corbea+u` en Unigram. Le jouet Unigram segmente `chanté` en six caractères ; l’entraîneur réel pourra choisir d’autres pièces.

### Exercice 2 — Le budget de l'Unigram

Reprendre les candidats après trois passes, élaguer à 60, 75, 93, 110 et 130 puis réestimer deux fois. Rapporter taille effective et tokens/mot. Deux jeux de budgets contrôlent les résultats contre une référence de calcul.

In [8]:
Dictionary<int,(int size,double rate)> CourbeBudget(int[] budgets,Dictionary<string,int> candidats){ // TODO : élaguer, réestimer deux fois et mesurer chaque budget
 return null;
}
(int size,double rate) ReferenceBudget(int k,Dictionary<string,int> candidats){var c=Refit(Prune(candidats,k),2);var l=c.ToDictionary(x=>x.Key,x=>Math.Log(x.Value/(double)c.Values.Sum()));return(c.Count,words.Sum(w=>Viterbi(w,l).Length)/(double)words.Length);}
bool ValideCourbe(int[] budgets,Dictionary<string,int> candidats,Dictionary<int,(int size,double rate)> result){if(result==null||!result.Keys.OrderBy(x=>x).SequenceEqual(budgets.OrderBy(x=>x)))return false;return budgets.All(k=>{var v=result[k];var expected=ReferenceBudget(k,candidats);return v.size>0&&double.IsFinite(v.rate)&&v.rate>0&&v.size==expected.size&&Math.Abs(v.rate-expected.rate)<1e-9;});}
int[] fixtureBudgets={60,75};var fixture=CourbeBudget(fixtureBudgets,fitted);
if(fixture==null)Console.WriteLine("Exercice 2 à compléter.");else{int[] autres={93,110,130};var second=CourbeBudget(autres,fitted);if(!ValideCourbe(fixtureBudgets,fitted,fixture)||!ValideCourbe(autres,fitted,second))Console.WriteLine("Fixtures incorrectes : vérifier élagage, deux passes et taille effective.");else {int[] budgets={60,75,93,110,130};var actual=CourbeBudget(budgets,fitted);if(!ValideCourbe(budgets,fitted,actual))Console.WriteLine("Résultat incorrect.");else foreach(var x in actual)Console.WriteLine($"budget {x.Key} : {x.Value.size} pièces, {x.Value.rate:F3} tokens/mot");}}

Exercice 2 à compléter.


## 5. Trois familles, une facture

Mots, caractères et trois familles sont comparés sur les 299 mots ; les chunks ont une capacité de 128 tokens. BPE et WP utilisent 94 symboles contre 93 pièces pour Unigram : budgets voisins, non identiques.

In [9]:
var rows=new[]{(name:"mots",v:words.Distinct().Count(),n:words.Length),(name:"caractères",v:chars.Count-1,n:words.Sum(w=>w.Length)),(name:"BPE",v:bpe.Count+chars.Count,n:bpeCount),(name:"WordPiece",v:wp.Count+chars.Count,n:wpCount),(name:"Unigram",v:finalCounts.Count,n:ugCount)};foreach(var r in rows)Console.WriteLine($"{r.name,-12} vocab={r.v,3} tokens={r.n,4} taux={r.n/(double)words.Length:F3} chunks@128={(r.n+127)/128}");

mots         vocab=188 tokens= 299 taux=1.000 chunks@128=3


caractères   vocab= 33 tokens=1313 taux=4.391 chunks@128=11


BPE          vocab= 94 tokens= 871 taux=2.913 chunks@128=7


WordPiece    vocab= 94 tokens=1099 taux=3.676 chunks@128=9


Unigram      vocab= 93 tokens= 850 taux=2.843 chunks@128=7


### Lecture du résultat

Le tableau donne 871 tokens pour BPE, 1099 pour WordPiece et 850 pour Unigram, avec respectivement 7, 9 et 7 chunks à capacité 128 : 9 contre 7 fait deux chunks supplémentaires. Les budgets sont voisins (94 contre 93), et ce classement concerne ces fables.

In [10]:
record SvgView(string Content);
Formatter.Register<SvgView>((v,w)=>w.Write(v.Content),"text/html");string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);string Escape(string s)=>System.Security.SecurityElement.Escape(s);
string Bars(string title,string[] labels,double[][] series,string[] legends){double max=series.SelectMany(x=>x).Max()*1.2;var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='800' height='430'><rect width='800' height='430' fill='white'/><path d='M75 55 V340 H760' stroke='black' fill='none'/>");void T(double x,double y,string t,string col="black")=>sb.Append($"<text x='{F(x)}' y='{F(y)}' font-size='13' fill='{col}'>{Escape(t)}</text>");T(190,23,title);T(4,40,"tokens/mot");for(int i=0;i<=5;i++){double y=340-i*57;sb.Append($"<path d='M70 {F(y)} H760' stroke='#ddd'/>");T(5,y+4,F(max*i/5));}string[] colors={"#2267a7","#b7352d","#278149","#986d21","#777777"};double group=650.0/labels.Length;for(int i=0;i<labels.Length;i++){double x=85+i*group;T(x,365,labels[i]);for(int j=0;j<series.Length;j++){double width=Math.Min(48,(group-12)/series.Length),v=series[j][i],height=v/max*285;sb.Append($"<rect x='{F(x+j*width)}' y='{F(340-height)}' width='{F(width-3)}' height='{F(height)}' fill='{colors[j]}'/>");T(x+j*width,335-height,F(v),colors[j]);}}for(int j=0;j<legends.Length;j++)T(85+j*180,405,legends[j],colors[j]);sb.Append("</svg>");return sb.ToString();}
var svgFamilies=Bars("Coût par famille",rows.Select(x=>x.name).ToArray(),new[]{rows.Select(x=>x.n/(double)words.Length).ToArray()},new[]{"tokens par mot"});display(new SvgView(svgFamilies));Console.WriteLine("Comparaison des cinq familles, tokens moyens par mot.");

Coût par famille tokens/mot 0 1.054 2.108 3.162 4.216 5.27 mots 1 caractères 4.391 BPE 2.913 WordPiece 3.676 Unigram 2.843 tokens par mot

Comparaison des cinq familles, tokens moyens par mot.


### Lecture du graphique

Les barres portent les mêmes moyennes que le tableau ; le contexte de comparaison est celui des fables et des budgets voisins.

## 6. Parité contre la bibliothèque réelle

`tokenizers` entraîne réellement WordPiece et Unigram au budget 93 sur les mêmes mots. Python.NET ne pilote que ces bibliothèques et leurs méthodes ; les calculs pédagogiques restent en C#.

In [11]:
string pyDll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");if(string.IsNullOrWhiteSpace(pyDll)||!File.Exists(pyDll))throw new Exception("Configurer PYTHONNET_PYDLL et PYTHONPATH avant le kernel : voir README.");if (!PythonEngine.IsInitialized)
{
    Runtime.PythonDLL = pyDll;
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}
else if (Path.GetFullPath(Runtime.PythonDLL) != Path.GetFullPath(pyDll))
    throw new InvalidOperationException("Python.NET utilise déjà une autre bibliothèque : redémarrer le noyau avec la configuration du README.");
PyObject wpReal=null,ugReal=null;int realWpSize=0,realUgSize=0,realWpCount=0,realUgCount=0;
using(Py.GIL()){
 using var module=Py.Import("tokenizers");using var modelsModule=Py.Import("tokenizers.models");using var trainersModule=Py.Import("tokenizers.trainers");dynamic tokenizers=module,models=modelsModule,trainers=trainersModule;
 using var pyWords=new PyList(words.Select(w=>(PyObject)new PyString(w)).ToArray());using var special=new PyList();
 dynamic w=tokenizers.Tokenizer(models.WordPiece(unk_token:"[UNK]"));w.train_from_iterator(pyWords,trainer:trainers.WordPieceTrainer(vocab_size:93,min_frequency:2,show_progress:false,special_tokens:special));
 dynamic u=tokenizers.Tokenizer(models.Unigram());u.train_from_iterator(pyWords,trainer:trainers.UnigramTrainer(vocab_size:93,show_progress:false,special_tokens:special));wpReal=(PyObject)w;ugReal=(PyObject)u;
 realWpSize=((PyObject)w.get_vocab_size()).As<int>();realUgSize=((PyObject)u.get_vocab_size()).As<int>();foreach(var word in words){realWpCount+=(int)((PyObject)w.encode(word).ids).Length();realUgCount+=(int)((PyObject)u.encode(word).ids).Length();}
 Console.WriteLine($"CPython {PythonEngine.Version} ; tokenizers {tokenizers.__version__} ; WordPiece vocabulaire {realWpSize}, {realWpCount} tokens ({realWpCount/(double)words.Length:F3}/mot) ; Unigram vocabulaire {realUgSize}, {realUgCount} tokens ({realUgCount/(double)words.Length:F3}/mot).");
 foreach(var word in testWords)Console.WriteLine($"{word,-12} jouet BPE [{string.Join("+",Encode(word,bpe))}] WP [{string.Join("+",Encode(word,wp))}] UG [{string.Join("+",Viterbi(word,ugLog))}] | Rust WP {w.encode(word).tokens} | Rust UG {u.encode(word).tokens}");
 try{w.encode("strawberry");throw new Exception("Mot inconnu accepté malgré l’absence de [UNK].");}catch(PythonException e) when(e.Message.Contains("Missing [UNK]")){Console.WriteLine("Mot hors corpus rejeté par WordPiece sans [UNK] réservé : "+e.Message.Split('\n')[0]);}
}

CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] ; tokenizers 0.23.3 ; WordPiece vocabulaire 93, 979 tokens (3.274/mot) ; Unigram vocabulaire 93, 812 tokens (2.716/mot).


corbeau      jouet BPE [c+orbeau] WP [corb+e+a+u] UG [corbea+u] | Rust WP ['c', '##or', '##b', '##eau'] | Rust UG ['co', 'rb', 'e', 'au']


vous         jouet BPE [vous] WP [vous] UG [vous] | Rust WP ['v', '##ous'] | Rust UG ['vo', 'us']


chanté       jouet BPE [ch+an+t+é] WP [chan+té] UG [c+h+a+n+t+é] | Rust WP ['ch', '##ant', '##é'] | Rust UG ['chant', 'é']


prêteuse     jouet BPE [pr+êt+eu+se] WP [p+r+êt+e+u+s+e] UG [p+r+ê+te+u+se] | Rust WP ['p', '##r', '##ê', '##t', '##eu', '##s', '##e'] | Rust UG ['p', 'r', 'ête', 'us', 'e']


renard       jouet BPE [renard] WP [r+e+n+a+r+d] UG [renard] | Rust WP ['r', '##en', '##ar', '##d'] | Rust UG ['ren', 'ar', 'd']


fromage      jouet BPE [f+ro+m+age] WP [fromag+e] UG [fromag+e] | Rust WP ['f', '##r', '##o', '##m', '##age'] | Rust UG ['f', 'ro', 'm', 'age']


dépourvue    jouet BPE [dé+p+our+v+ue] WP [dép+o+u+r+v+u+e] UG [d+é+p+o+u+r+v+u+e] | Rust WP ['d', '##é', '##p', '##our', '##v', '##ue'] | Rust UG ['d', 'é', 'p', 'our', 'v', 'ue']


patience     jouet BPE [pa+ti+en+c+e] WP [p+a+t+i+e+n+c+e] UG [p+a+t+i+e+n+ce] | Rust WP ['p', '##a', '##t', '##i', '##en', '##c', '##e'] | Rust UG ['pa', 't', 'ien', 'ce']


Mot hors corpus rejeté par WordPiece sans [UNK] réservé : WordPiece error: Missing [UNK] token from the vocabulary


### Lecture du résultat

Les entraîneurs Rust donnent 979 tokens WordPiece et 812 Unigram pour 93 pièces chacun, contre 1099 et 850 pour les jouets. Leurs fragments diffèrent : `chanté` donne notamment `chant+é` avec Unigram Rust. Le trainer source ne réserve pas `[UNK]` : certains mots inconnus sont rejetés.

### DistilBERT et Qwen réels

DistilBERT minuscule et retire les accents. Qwen utilise le vocabulaire et les fusions du modèle, sa regex de pré-tokenisation et la normalisation NFC avant l'encodage byte-level. Les fichiers publics sont épinglés et leurs empreintes SHA-256 contrôlées.

In [12]:
string qPath=null,qConfigPath=null,bertPath=null,bertJson=null;PyObject bertModel=null;
using(Py.GIL()){using var hub=Py.Import("huggingface_hub");dynamic hf=hub;qPath=((PyObject)hf.hf_hub_download("Qwen/Qwen3.5-0.8B","tokenizer.json",revision:"2fc06364715b967f1860aea9cf38778875588b17")).As<string>();qConfigPath=((PyObject)hf.hf_hub_download("Qwen/Qwen3.5-0.8B","tokenizer_config.json",revision:"2fc06364715b967f1860aea9cf38778875588b17")).As<string>();bertPath=((PyObject)hf.hf_hub_download("distilbert/distilbert-base-uncased","vocab.txt",revision:"12040accade4e8a0f71eabdb258fecc2e7e948be")).As<string>();bertJson=((PyObject)hf.hf_hub_download("distilbert/distilbert-base-uncased","tokenizer.json",revision:"12040accade4e8a0f71eabdb258fecc2e7e948be")).As<string>();using var toks=Py.Import("tokenizers");dynamic tok=toks;bertModel=(PyObject)tok.Tokenizer.from_file(bertJson);Console.WriteLine($"tokenizers {tok.__version__} ; huggingface-hub {hf.__version__}");}
void CheckHash(string path,string expected){string actual=Convert.ToHexString(SHA256.HashData(File.ReadAllBytes(path))).ToLowerInvariant();if(actual!=expected)throw new Exception("Empreinte invalide : "+Path.GetFileName(path));Console.WriteLine($"{Path.GetFileName(path)} SHA-256 {actual}");}
CheckHash(qPath,"5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42");CheckHash(qConfigPath,"49e2b6e395f959f077f1e992b338919c0d4a9732fc6e613995e06557f843500c");CheckHash(bertPath,"07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3");CheckHash(bertJson,"ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98");
var root=JsonDocument.Parse(File.ReadAllText(qPath)).RootElement;var model=root.GetProperty("model");var vocab=model.GetProperty("vocab").EnumerateObject().ToDictionary(x=>x.Name,x=>x.Value.GetInt32());var merges=model.GetProperty("merges").EnumerateArray().Select(x=>x.GetString()).ToList();var config=JsonDocument.Parse(File.ReadAllText(qConfigPath)).RootElement;var specials=config.GetProperty("added_tokens_decoder").EnumerateObject().ToDictionary(x=>x.Value.GetProperty("content").GetString(),x=>int.Parse(x.Name));var pattern=root.GetProperty("pre_tokenizer").GetProperty("pretokenizers")[0].GetProperty("pattern").GetProperty("Regex").GetString();var qwen=BpeTokenizer.Create(new BpeOptions(vocab){Merges=merges,SpecialTokens=specials,ByteLevel=true,PreTokenizer=new RegexPreTokenizer(new Regex(pattern),specials)});
var bertSdk=BertTokenizer.Create(bertPath,new BertOptions{LowerCaseBeforeTokenization=true,RemoveNonSpacingMarks=true});int[] Qwen(string text)=>qwen.EncodeToIds(text.Normalize(NormalizationForm.FormC)).ToArray();
Console.WriteLine($"DistilBERT vocabulaire 30522 ; Qwen vocabulaire {vocab.Count} ; NFC avant byte-level.");
using(Py.GIL()){dynamic bert=bertModel;foreach(string w in new[]{"corbeau","dépourvue","chanté","strawberry","🙂"})Console.WriteLine($"{w,-12} BERT réel {bert.encode(w,add_special_tokens:false).tokens} ids {bert.encode(w,add_special_tokens:false).ids} | Qwen [{string.Join(",",Qwen(w))}] | BERT SDK [{string.Join(",",bertSdk.EncodeToIds(w,addSpecialTokens:false))}]");}

tokenizers 0.23.3 ; huggingface-hub 2.4.0


tokenizer.json SHA-256 5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42


tokenizer_config.json SHA-256 49e2b6e395f959f077f1e992b338919c0d4a9732fc6e613995e06557f843500c


vocab.txt SHA-256 07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3


tokenizer.json SHA-256 ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98


DistilBERT vocabulaire 30522 ; Qwen vocabulaire 248044 ; NFC avant byte-level.


... (2 outputs left out)

chanté       BERT réel ['chant', '##e'] ids [16883, 2063] | Qwen [11345,933] | BERT SDK [16883,2063]


strawberry   BERT réel ['strawberry'] ids [16876] | Qwen [485,654,14911] | BERT SDK [16876]


🙂           BERT réel ['[UNK]'] ids [100] | Qwen [169171] | BERT SDK []



warning CS1701: Assuming assembly reference 'System.Text.RegularExpressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'Microsoft.ML.Tokenizers' matches identity 'System.Text.RegularExpressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Text.RegularExpressions', you may need to supply runtime policy



### Lecture du résultat

Le pipeline DistilBERT produit `de+##pour+##vu+##e` pour `dépourvue` : l’accent a disparu avant la découpe. Sur `🙂`, la référence donne `[UNK]` (ID 100) tandis que le SDK BERT .NET renvoie une liste vide ; aucune équivalence Unicode universelle n’est affirmée. Qwen natif utilise la regex et le NFC du modèle.

## 7. L'effet multilingue sur le coût

Trois traductions d'une même ligne sont comparées avec Qwen sur le texte normalisé joint et BPE jouet mot à mot. Trois phrases ne permettent pas d'établir une propriété générale des langues.

In [13]:
var lines=new Dictionary<string,string>{{"fr","Le corbeau sur un arbre perché tient un fromage dans son bec"},{"en","The crow perched on a tree holds a cheese in his beak"},{"es","El cuervo posado en un árbol tiene un queso en su pico"}};
(int words,int qwen,int toy) MeasureLine(string text){var ws=Words(text);return(ws.Length,Qwen(string.Join(" ",ws)).Length,ws.Sum(w=>Encode(w,bpe).Length));}
var langRows=lines.ToDictionary(x=>x.Key,x=>MeasureLine(x.Value));foreach(var kv in langRows){var r=kv.Value;Console.WriteLine($"{kv.Key} : {r.words} mots, Qwen {r.qwen} ({r.qwen/(double)r.words:F2}/mot), BPE jouet {r.toy} ({r.toy/(double)r.words:F2}/mot), ratio {r.toy/(double)r.qwen:F2}");}

fr : 12 mots, Qwen 13 (1.08/mot), BPE jouet 30 (2.50/mot), ratio 2.31


en : 12 mots, Qwen 14 (1.17/mot), BPE jouet 35 (2.92/mot), ratio 2.50


es : 12 mots, Qwen 14 (1.17/mot), BPE jouet 36 (3.00/mot), ratio 2.57


### Lecture du résultat

Pour les trois phrases, Qwen donne 13, 14 et 14 tokens en français, anglais et espagnol ; le BPE jouet en donne 30, 35 et 36, soit 2,50, 2,92 et 3,00 tokens/mot. Qwen encode le texte joint, le jouet encode chaque mot. Ces trois phrases ne démontrent pas une hiérarchie universelle entre langues.

In [14]:
var svgLanguages=Bars("Trois traductions",langRows.Keys.Select(x=>x.ToUpperInvariant()).ToArray(),new[]{langRows.Values.Select(x=>x.qwen/(double)x.words).ToArray(),langRows.Values.Select(x=>x.toy/(double)x.words).ToArray()},new[]{"Qwen réel","BPE jouet"});display(new SvgView(svgLanguages));Console.WriteLine("Barres : Qwen réel et BPE jouet, tokens moyens par mot.");

Trois traductions tokens/mot 0 0.72 1.44 2.16 2.88 3.6 FR 1.083 2.5 EN 1.167 2.917 ES 1.167 3 Qwen réel BPE jouet

Barres : Qwen réel et BPE jouet, tokens moyens par mot.


### Lecture du graphique

Les barres expriment les mêmes taux que le tableau. Un vocabulaire multilingue ne garantit pas un coût identique pour toutes les langues ou tous les textes.

### Exercice 3 — Prédire puis vérifier : l'allemand

Ajouter « Der Rabe, der auf einem Baum saß, hielt einen Käse im Schnabel », prédire les classements puis calculer les deux compteurs pour les quatre langues. Aucun ordre n'est imposé. Deux fixtures vérifient les valeurs, dimensions et appels.

In [15]:
Dictionary<string,(int qwen,int toy)> MesurerLangues(Dictionary<string,string> textes,Func<string,(int qwen,int toy)> mesure){ // TODO : appliquer mesure à chaque texte
 return null;
}
var testTextes=new Dictionary<string,string>{{"fr","aa"},{"en","bbbb"},{"de","ccc"}};var appels=new List<string>();var fLang=MesurerLangues(testTextes,t=>{appels.Add(t);return(t.Length,t.Length+2);});
if(fLang==null)Console.WriteLine("Exercice 3 à compléter.");else{var appels2=new List<string>();var fLang2=MesurerLangues(testTextes,t=>{appels2.Add(t);return(t.Length*3,t.Length*5);});bool ok=fLang2!=null && fLang.Keys.OrderBy(x=>x).SequenceEqual(testTextes.Keys.OrderBy(x=>x)) && fLang2.Keys.OrderBy(x=>x).SequenceEqual(testTextes.Keys.OrderBy(x=>x)) && appels.OrderBy(x=>x).SequenceEqual(testTextes.Values.OrderBy(x=>x)) && appels2.OrderBy(x=>x).SequenceEqual(testTextes.Values.OrderBy(x=>x)) && testTextes.All(x=>fLang[x.Key]==(x.Value.Length,x.Value.Length+2)&&fLang2[x.Key]==(x.Value.Length*3,x.Value.Length*5));if(!ok)Console.WriteLine("Fixtures incorrectes.");else{var textes=new Dictionary<string,string>(lines){{"de","Der Rabe, der auf einem Baum saß, hielt einen Käse im Schnabel"}};var result=MesurerLangues(textes,t=>{var r=MeasureLine(t);return(r.qwen,r.toy);});if(result==null||!result.Keys.OrderBy(x=>x).SequenceEqual(textes.Keys.OrderBy(x=>x))||textes.Any(x=>result[x.Key]!=(MeasureLine(x.Value).qwen,MeasureLine(x.Value).toy)))Console.WriteLine("Résultat incorrect.");else foreach(var kv in result)Console.WriteLine($"{kv.Key} Qwen={kv.Value.qwen} jouet={kv.Value.toy}");}}

Exercice 3 à compléter.


## Exercices

- [Autre corpus](#exercice-1--wordpiece-sur-un-autre-corpus) : entraîner et comparer les fusions BPE et WordPiece.
- [Budget Unigram](#exercice-2--le-budget-de-lunigram) : mesurer les tailles effectives et les tokens/mot pour cinq budgets.
- [Allemand](#exercice-3--prédire-puis-vérifier--lallemand) : prédire puis mesurer les deux compteurs sur quatre langues.

## À retenir

Le critère de sélection modifie les pièces, et le budget de vocabulaire ne suffit pas à caractériser le coût. Les entraîneurs Rust sont de vrais points de comparaison ; la normalisation et le pré-tokenizer font partie du modèle.

## Vérifiez votre compréhension

<details><summary>Pourquoi 9 contre 7 chunks fait-il deux chunks supplémentaires ?</summary>La différence est 9 − 7 = 2, après arrondi de chaque total au multiple de 128.</details>

<details><summary>Que signifie la limite de l'implémentation BERT sur un emoji ?</summary>Elle interdit d'affirmer une équivalence Unicode universelle avec le pipeline de référence.</details>

## Pour aller plus loin

Comparer d'autres corpus et langues, mesurer des segmentations alternatives Unigram, puis poursuivre vers le [stockage vectoriel](05-Stockage-Vectoriel.ipynb). Documentation : [Hugging Face tokenizers](https://huggingface.co/docs/tokenizers).